<a href="https://colab.research.google.com/github/VictoriaUsman/AI_dev/blob/main/chatsimp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

In [8]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_size, embed_size)
        self.rnn = nn.GRU(embed_size, hidden_size)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, hidden = self.rnn(embedded)
        return hidden

In [9]:
class Decoder(nn.Module):
    def __init__(self, output_size, embed_size, hidden_size):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_size, embed_size)
        self.rnn = nn.GRU(embed_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        x = x.unsqueeze(0)
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden

In [10]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size)

        hidden = self.encoder(src)

        x = trg[0]  # start token

        for t in range(1, trg_len):
            output, hidden = self.decoder(x, hidden)
            outputs[t] = output

            best_guess = output.argmax(1)
            x = trg[t] if torch.rand(1).item() < teacher_forcing_ratio else best_guess

        return outputs

In [11]:
# Vocabulary sizes (dummy example)
INPUT_SIZE = 10
OUTPUT_SIZE = 10

encoder = Encoder(INPUT_SIZE, embed_size=8, hidden_size=16)
decoder = Decoder(OUTPUT_SIZE, embed_size=8, hidden_size=16)

model = Seq2Seq(encoder, decoder)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)